Group: Yifan Liu, Arvind Sankar, Leny Pan, Lucas Schmitt

Netid: yifanl25, asankar2, lenypan2, lucass6

Email: yifanl25@illinois.edu, asankar2@illinois.edu, lenypan2@illinois.edu, lucass6@illinois.edu

Date: 11/17/2025

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
users = pd.read_csv('users.dat', delimiter='::', header = None, names = ["UserID", "Gender", "Age", "Occupation", "Zip-code"], engine="python")
age_dummies = pd.get_dummies(users['Age'])
age_dummies.columns = ["Under 18", "18-24", "25-34", "35-44", "45-49", "50-55", "56+"]
occupation_dummies = pd.get_dummies(users['Occupation'])
occupation_dummies.columns = ["other", "academic/educator", "artist", "clerical/admin", "college/grad student", "customer service", "doctor/health care",
                              "executive/managerial", "farmer", "homemaker", "K-12 student", "lawyer", "programmer", "retired", "sales/marketing",
                              "scientist", "self-employed", "technician/engineer", "tradesman/craftsman", "unemployed", "writer"]
users = pd.concat([users[["UserID", "Gender", "Zip-code"]], occupation_dummies, age_dummies] , axis = 1)

ratings = pd.read_csv('ratings.dat', delimiter='::', header = None, names = ["UserID", "MovieID", "Rating", "Timestamp"], engine="python")

movies = pd.read_csv('movies.dat', delimiter='::', header = None, names = ["MovieID","Title","Genres"], encoding='latin-1', engine="python")
genre_dummies = movies['Genres'].str.get_dummies(sep='|')
movies = pd.concat([movies[['MovieID', 'Title']], genre_dummies], axis=1)

In [ ]:
ratings_movie_ids = ratings['MovieID'].unique()
movies = movies[movies['MovieID'].isin(ratings_movie_ids)]

#print(ratings_movie_ids)

In [ ]:
print(len(movies))
print(ratings['MovieID'].nunique())

3706
3706


After loading and pre-processing the data as needed, consider the following tasks.

(A) Recommend the top 10 “most popular ” movies. The term “most popular” is open
to interpretation from each group. So, it is important to clearly state how you define the
popularity of a movie. The recommendation you make should display the top 10 movie
titles, and information related to the criteria you used - e.g. if you recommend the top 10
movies based on ratings for movies with at least 1,000 ratings, report these numbers.

----
We define most popular movies as movies with the most ratings overall. "Popular" does not mean best or highest rated. Our metric of popularity is based off of how many people have shared their opinion on a movie. The code below gets the top 10 movies with the most ratings.

In [ ]:
import copy
rate_copy = copy.deepcopy(ratings)
movies_copy = copy.deepcopy(movies)

In [ ]:

def get_top_n_movies(n):

  #print(rate_copy.head())

  rating_counts = rate_copy.groupby('MovieID')['Rating'].count()
  rating_counts_renamed = rating_counts.rename('NumRatings')
  popular_movies = movies_copy.merge(rating_counts_renamed, left_on='MovieID', right_index=True)


  top = popular_movies.nlargest(n,'NumRatings')
  return top['Title'], top['MovieID'], top['NumRatings']



In [ ]:
title_out, ids_out, ratings_out = get_top_n_movies(10)

In [ ]:
for i in range(10):
  print(f"Title: {title_out.iloc[i]:<60} MovieID: {str(ids_out.iloc[i]):<8} NumRatings: {str(ratings_out.iloc[i]):<6}")

Title: American Beauty (1999)                                       MovieID: 2858     NumRatings: 3428  
Title: Star Wars: Episode IV - A New Hope (1977)                    MovieID: 260      NumRatings: 2991  
Title: Star Wars: Episode V - The Empire Strikes Back (1980)        MovieID: 1196     NumRatings: 2990  
Title: Star Wars: Episode VI - Return of the Jedi (1983)            MovieID: 1210     NumRatings: 2883  
Title: Jurassic Park (1993)                                         MovieID: 480      NumRatings: 2672  
Title: Saving Private Ryan (1998)                                   MovieID: 2028     NumRatings: 2653  
Title: Terminator 2: Judgment Day (1991)                            MovieID: 589      NumRatings: 2649  
Title: Matrix, The (1999)                                           MovieID: 2571     NumRatings: 2590  
Title: Back to the Future (1985)                                    MovieID: 1270     NumRatings: 2583  
Title: Silence of the Lambs, The (1991)                

(B) Make a recommendation based on Item-Based Collaborative Filtering (IBCF) by
writing your own IBCF function following the steps below:
1. Create the rating matrix by R, where the rows correspond to users and the columns
to movies. Normalize R by centering each row, i.e. subtract the row means from
each row of the matrix. The matrix is sparse, so the row means should be computed
based on the non-NA entries.

In [ ]:
item_matrix = pd.pivot_table(data = ratings, index='UserID', columns='MovieID', values='Rating')

num_movies = len(item_matrix.iloc[0])
#print(numMovies)

average_mat = item_matrix.copy()

# subtract row means from each row of the matrix which achieves centering (eliminating user bias)
average_mat = item_matrix.sub(item_matrix.mean(axis=1), axis=0)

average_mat.head() # size is users x 3706

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
UserID,,,,,,,,,,,,,,,,,,,,,
1,0.811321,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,-1.146465,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2. Compute the cosine similarity for all the movies. You can ignore similarities computed based on less than 3 user ratings. You can also use the (1 + cos)/2 transformation to ensure than the similarity will be between 0 and 1. Keep in mind that
you may have NAs when a pair off movies (for example) has been rated by 0, 1, or
2 users, or when the denominator in the similarity metric is zero.

In [382]:
from sklearn.metrics.pairwise import cosine_similarity

# Data Preprocessing
# First remove similarities computed based on less than 3 user ratings
average_mat = average_mat.drop(average_mat.columns[np.where(average_mat.count() < 3)], axis=1)
# Then fill in nan values with 0.
average_mat_filled = average_mat.fillna(0)

print(average_mat_filled.head()) # Size is user x 3503
# We want to make a cosine similarity matrix of movies, based on ratings the
# users have given. Our "rows" are currently user's ratings and the columns are
# movie ids, so we need to find cosine similarity of the transpose of our
# matrix.
#
# Doing it this way takes advantage of sklearn vectorized matrix operations.
movie_similarity_mat = cosine_similarity(average_mat_filled.T)
movie_similarity_mat = (movie_similarity_mat + 1)/2

print("-"*80)
print(movie_similarity_mat)

MovieID      1     2     3     4     5         6     7     8     9     10    \
UserID                                                                        
1        0.811321   0.0   0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0   
2        0.000000   0.0   0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0   
3        0.000000   0.0   0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0   
4        0.000000   0.0   0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0   
5        0.000000   0.0   0.0   0.0   0.0 -1.146465   0.0   0.0   0.0   0.0   

MovieID  ...  3943  3944  3945  3946  3947  3948  3949  3950  3951  3952  
UserID   ...                                                              
1        ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2        ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3        ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4        ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.

In [383]:
# crucial mapping to the reduced matrix 3503
remaining_movie_ids = average_mat_filled.columns.values

movie_id_to_pos = {movie_id: idx for idx, movie_id in enumerate(remaining_movie_ids)}



3. For the similarity matrix in (2), sort the non-NA similarity measures and keep the top 30, setting the rest to NA. Display the pairwise similarity values for this new matrix for the following movies:


*   “Toy Story (1995)”
*   “GoldenEye (1995)”


*   “Liar Liar (1997)”
*   “Lost World: Jurassic Park (1997)”

*   “Sixth Sense, The (1999)”


In [384]:
sorted_similarity_mat_i = np.argsort(movie_similarity_mat, axis=1)
sorted_similarity_mat_i = np.flip(sorted_similarity_mat_i, axis=1)
#print(sorted_similarity_mat_i)
top_30_indices = sorted_similarity_mat_i[:, :31]

# mask out the top 30 indices. Everything else becomes nan
mask = np.zeros_like(movie_similarity_mat, dtype=bool)
rows = np.arange(movie_similarity_mat.shape[0])
for i in rows:
  mask[i, top_30_indices[i]] = True


top_30_mov_sim_mat = np.where(mask, movie_similarity_mat, np.nan)


# fill the diagonals with nans to remove self recommendations
np.fill_diagonal(top_30_mov_sim_mat, np.nan)
#count_non_nan = np.sum(~np.isnan(top_30_mov_sim_mat[7]))
#print(count_non_nan)


#print(top_30_mov_sim_mat.shape)



selected = ['Toy Story (1995)',  # indices: [1, 10, 1485, 1544, 2762]
            'GoldenEye (1995)',
            'Liar Liar (1997)',
            'Lost World: Jurassic Park',
            'Sixth Sense, The (1999)',
            #'Postino, Il (The Postman) (1994)',
            ]

movie_ids = []
movie_positions = []
for title in selected:
    match = movies[movies['Title'].str.contains(title, regex=False)]
    movie_id = int(match['MovieID'].values[0])
    movie_ids.append(movie_id)

    if movie_id in movie_id_to_pos:
        movie_positions.append(movie_id_to_pos[movie_id])

#print(movie_ids)
#print(movie_positions)
#print(top_30_mov_sim_mat)

subset = top_30_mov_sim_mat[np.ix_(movie_positions, movie_positions)]

pairwise_sim = pd.DataFrame(subset, index=movie_ids, columns=movie_ids)

print(pairwise_sim)

         1         10    1485  1544     2762
1         NaN       NaN   NaN   NaN  0.62578
10        NaN       NaN   NaN   NaN      NaN
1485      NaN  0.544694   NaN   NaN      NaN
1544      NaN       NaN   NaN   NaN      NaN
2762  0.62578       NaN   NaN   NaN      NaN


4. Write your own IBCF function:


*   Input: newuser, i.e. a vector containing ratings for all the movies.
Many of the entries will be NA, while the non-NA values should be ratings 1, 2,
3, 4, 5 - the star ratings.

*   Output: The top 10 movies recommendation for this new user.
If fewer than 10 predictions are non-NA, select the remaining movies based on
the popularity defined in part (A), prioritizing the most popular ones and excluding those already rated by the user. Here, you may want to save the ranking
of all movies (based on popularity) in a separate file to avoid recomputing the
ranking each time.

*   The function you write should load the similarity matrix and use it to compute
predictions for movies that have not been rated by the new user yet, using the
formula from the notes/slides.


In [ ]:
def IBCF(new_user: pd.Series):
    movie_sum = np.zeros(max(remaining_movie_ids) + 1)
    #print(movie_sum.shape)

    unrated, rated = {}, {}
    for movie_id, val in zip(list(new_user.index), new_user.values):
      if movie_id not in remaining_movie_ids:
        continue
      if np.isnan(val):
        unrated[movie_id] = val
      else:
        rated[movie_id] = val


    for unrated_i in unrated:
        numerator = 0
        denominator = 0
        for rated_i, score in rated.items():
            sim = top_30_mov_sim_mat[movie_id_to_pos[unrated_i], movie_id_to_pos[rated_i]]

            if np.isnan(sim):
                continue

            numerator += sim * score
            denominator += sim

        if denominator != 0:
            movie_sum[unrated_i] = numerator / denominator
        else:
            movie_sum[unrated_i] = np.nan

    count_non_nan = np.sum(~np.isnan(movie_sum))
    #print(count_non_nan)

    top_10_indices = np.argsort(movie_sum)[::-1][:10]
    top_10_recommendations = []
    for index in top_10_indices:
        title = movies.loc[movies['MovieID'] == index, 'Title'].values[0]
        top_10_recommendations.append(title)

    i = 0
    while len(top_10_recommendations) < 10:
      if title_out.iloc[i] in top_10_recommendations:
          i += 1
          continue
      top_10_recommendations.append(title_out.iloc[i])
      i += 1

    return top_10_recommendations

5. Test your function by displaying the top 10 recommendations for the following two
users:

*   User in row 1500 from the rating matrix.
*   A hypothetical user who rates movie “Star Wars: Episode IV - A New Hope
(1977)” with 5 stars and movie “Independence Day (ID4) (1996)” with 4 stars.





In [ ]:
user_1500 = item_matrix.iloc[1499]
print(user_1500)
#print(user_1500.shape)

res = IBCF(user_1500)


MovieID
1       NaN
2       NaN
3       NaN
4       NaN
5       NaN
       ... 
3948    5.0
3949    5.0
3950    NaN
3951    NaN
3952    NaN
Name: 1500, Length: 3706, dtype: float64


In [ ]:
print(res)

['Tin Cup (1996)', 'Dingo (1992)', 'One True Thing (1998)', 'Permanent Midnight (1998)', 'Ballad of Narayama, The (Narayama Bushiko) (1958)', 'Indecent Proposal (1993)', 'Maybe, Maybe Not (Bewegte Mann, Der) (1994)', 'Mortal Thoughts (1991)', 'Last Supper, The (1995)', 'Nothing But Trouble (1991)']


In [ ]:
selected = ['Star Wars: Episode IV - A New Hope (1977)',
            'Independence Day (ID4) (1996)',
            ]

movie_ids = []
for title in selected:
    match = movies[movies['Title'].str.contains(title, regex=False)]
    movie_id = int(match['MovieID'].values[0])
    movie_ids.append(movie_id)

print(movie_ids)


[260, 780]


In [ ]:
hypo_user = pd.Series([np.nan] * len(user_1500), dtype=float, index=user_1500.index)
hypo_user.iloc[movie_ids[0]] = 5
hypo_user.iloc[movie_ids[1]] = 4

print(hypo_user)

hypo_user_res = IBCF(hypo_user)

MovieID
1      NaN
2      NaN
3      NaN
4      NaN
5      NaN
        ..
3948   NaN
3949   NaN
3950   NaN
3951   NaN
3952   NaN
Length: 3706, dtype: float64


In [ ]:
print(hypo_user_res)

['Grateful Dead (1995)', 'Message to Love: The Isle of Wight Festival (1996)', 'Walkabout (1971)', 'Portrait of a Lady, The (1996)', 'Evita (1996)', 'Thieves (Voleurs, Les) (1996)', 'Mother (1996)', 'Whole Wide World, The (1996)', "Some Mother's Son (1996)", 'Hamlet (1996)']
